# Run RFI processing on selected products

In [ ]:
%pip install aiohttp requests
%load_ext autoreload
%autoreload 2

import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
import geopandas as gpd
from tqdm import tqdm
import pandas as pd
import numpy as np
import requests
import pathlib
import os
import json
import yaml
from pathlib import Path

from processing import pol_mean, read_item
from plotting import fetch_quicklook, plot_product, plot_result

In [ ]:
# Load the config.yaml to access the paths
cfg_file = Path("../config.yaml").resolve()
cfg = yaml.safe_load(cfg_file.read_text())

# Set the name of the path in the config.yaml
token_path = (cfg_file.parent / cfg["paths"]["token_file"]).resolve()
out_path = (cfg_file.parent / cfg["paths"]["output"]).resolve()
data_path = (cfg_file.parent / cfg["paths"]["data"]).resolve()

rfi_prod_path = out_path / cfg["files"]["rfi_products"]
gdf_path = out_path / cfg["files"]["gdf"]

print(f"Token Path: {token_path}")
print(f"Output Path: {out_path}")
print(f"List of RFI products Path: {rfi_prod_path}")
print(f"GeoDataframe of RFI products Path: {gdf_path}")

In [ ]:
# Load the list of products affected by RFI
with open(rfi_prod_path) as f:
    products_rfi = json.load(f)

# Load the GeoDataframe
gdf = gpd.read_parquet(gdf_path)

gdf.head()
print(len(gdf))
print(len(products_rfi))

In [ ]:
catalog_url = "https://catalog.maap.eo.esa.int/catalogue/"
catalog = Client.open(catalog_url)

# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID = "offline-token"
CLIENT_SECRET = "p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

if pathlib.Path(token_path).exists():
    with open(token_path, "rt") as f:
        OFFLINE_TOKEN = f.read().strip().replace("\n", "")
        url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
        data = {
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "grant_type": "refresh_token",
            "refresh_token": OFFLINE_TOKEN,
            "scope": "offline_access openid",
        }

        response = requests.post(url, data=data)
        response.raise_for_status()

        response_json = response.json()
        access_token = response_json.get("access_token")
        print("Access token retrieved successfully.")

        if not access_token:
            raise RuntimeError("Failed to retrieve access token from IAM response")

else:
    raise RuntimeError(f"No token file found in {os.getcwd()} !")

Select which products to explore:
- From the products_rfi.json which are converted to a dataframe. As an example we select the unmitigated proucts with an iso_impact larger than 10%
- Select any of the hard-coded list
- Conventional filtering by bbox, dates etc.

In [ ]:
print(len(gdf))
print(len(products_rfi))
rows = [
    {
        "product_id": p["id"],
        "rfiCorrectionFlag": p["rfiCorrectionFlag"],
        **p["properties"],
        **p["fm"],
    }
    for p in products_rfi
]
print(len(rows))
df_rfi = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["product_id"])
print(len(df_rfi))
for kind in ("pers", "iso"):
    df_rfi[f"{kind}_avgBW"] = pol_mean(df_rfi, kind, "avgBW")
    df_rfi[f"{kind}_linesPct"] = pol_mean(df_rfi, kind, "linesPct")
    df_rfi[f"{kind}_impact"] = df_rfi[f"{kind}_avgBW"] * df_rfi[f"{kind}_linesPct"] / 100
    df_rfi[f"{kind}_impact"] = df_rfi[f"{kind}_avgBW"] * df_rfi[f"{kind}_linesPct"] / 100

df_selected = df_rfi[df_rfi["rfiCorrectionFlag"] == "false"]

PRCT_IMPACT = 10.0
n_before = len(df_selected)
df_selected = df_selected[df_selected["iso_impact"] >= PRCT_IMPACT]
print(
    f"discarded {n_before - len(df_selected)} of {n_before} products "
    f"(iso_impact < {PRCT_IMPACT} or missing), {len(df_selected)} kept"
)

In [ ]:
# Middle east
middle_east = [
    "BIO_S1_SCS__1S_20260513T150807_20260513T150827_T_G01_M03_C07_T039_F134_02_DRE0O0",
    "BIO_S1_SCS__1S_20260513T150904_20260513T150924_T_G01_M03_C07_T039_F137_02_DRDZV9",
    "BIO_S1_SCS__1S_20260513T150826_20260513T150846_T_G01_M03_C07_T039_F135_02_DRE0HI",
    "BIO_S3_SCS__1S_20260829T150437_20260829T150457_T_G01_M04_C05_T039_F107_02_DWY0RR",
    "BIO_S3_SCS__1S_20260829T150359_20260829T150419_T_G01_M04_C05_T039_F105_02_DWY14T",
]
# Ecuador
ecuador = [
    "BIO_S3_SCS__1S_20260829T233255_20260829T233313_T_G01_M04_C05_T044_F162_02_DWYO99",
    "BIO_S3_SCS__1S_20260829T233236_20260829T233257_T_G01_M04_C05_T044_F161_02_DWYO8H",
    "BIO_S3_SCS__1S_20260829T233217_20260829T233238_T_G01_M04_C05_T044_F160_02_DWYO38",
    "BIO_S3_SCS__1S_20260829T233158_20260829T233219_T_G01_M04_C05_T044_F159_02_DWYO4X",
    "BIO_S3_SCS__1S_20260829T233139_20260829T233200_T_G01_M04_C05_T044_F158_02_DWYO5W",
    "BIO_S3_SCS__1S_20260829T233120_20260829T233141_T_G01_M04_C05_T044_F157_02_DWYO5M",
    "BIO_S3_SCS__1S_20260829T233101_20260829T233122_T_G01_M04_C05_T044_F156_02_DWYO3C",
    "BIO_S3_SCS__1S_20260829T233042_20260829T233103_T_G01_M04_C05_T044_F155_02_DWYOAL",
    "BIO_S3_SCS__1S_20260829T233023_20260829T233044_T_G01_M04_C05_T044_F154_02_DWYO9B",
    "BIO_S3_SCS__1S_20260829T233004_20260829T233025_T_G01_M04_C05_T044_F153_02_DWYOAY",
]

selected = [
    # Set 1, Region 1
    "BIO_S1_SCS__1S_20260915T104023_20260915T104044_T_G01_M05_C01_T021_F293_02_DXTLHJ",
    "BIO_S1_SCS__1S_20260918T104025_20260918T104046_T_G01_M05_C02_T021_F293_02_DXZAG1",
    "BIO_S1_SCS__1S_20260921T104028_20260921T104048_T_G01_M05_C03_T21_F293_02_DY4MWQ",
    "BIO_S1_SCS__1S_20260924T104030_20260924T104050_T_G01_M05_C04_T021_F293_02_DYA6ZC",
    "BIO_S1_SCS__1S_20260927T104032_20260927T104052_T_G01_M05_C05_T021_F293_02_DYFQMZ",
    "BIO_S1_SCS__1S_20260930T104034_20260930T104055_T_G01_M05_C06_T021_F293_02_DYLATM",
    "BIO_S1_SCS__1S_20260915T104101_20260915T104122_T_G01_M05_C01_T021_F295_02_DXTLHG",
    "BIO_S1_SCS__1S_20260918T104103_20260918T104124_T_G01_M05_C02_T021_F295_02_DXZAHN",
    "BIO_S1_SCS__1S_20260921T104106_20260921T104126_T_G01_M05_C03_T021_F295_02_DY4MZ4",
    "BIO_S1_SCS__1S_20260924T104108_20260924T104128_T_G01_M05_C04_T021_F295_02_DYA6ZD",
    "BIO_S1_SCS__1S_20260927T104110_20260927T104130_T_G01_M05_C05_T021_F295_02_DYFQOH",
    "BIO_S1_SCS__1S_20260930T104112_20260930T104133_T_G01_M05_C06_T021_F295_02_DYLAZI",
    # Set 1, Region 2
    "BIO_S1_SCS__1S_20260916T220549_20260916T220609_T_G01_M05_C01_T043_F184_02_DXW72Z",
    "BIO_S1_SCS__1S_20260919T220551_20260919T220611_T_G01_M05_C02_T043_F184_02_DY1SIO",
    "BIO_S1_SCS__1S_20260922T220553_20260922T220614_T_G01_M05_C03_T043_F184_02_DY79AQ",
    "BIO_S1_SCS__1S_20260925T220555_20260925T220616_T_G01_M05_C04_T043_F184_02_DYCSA3",
    "BIO_S1_SCS__1S_20260928T220558_20260928T220618_T_G01_M05_C05_T043_F184_02_DYICLW",
    # Set 2, Region 1
    "BIO_S1_SCS__1S_20260915T212528_20260915T212549_T_G01_M05_C01_T028_F160_02_DXUDDE",
    "BIO_S1_SCS__1S_20260918T212530_20260918T212551_T_G01_M05_C02_T028_F160_02_DY00H6",
    "BIO_S1_SCS__1S_20260921T212532_20260921T212553_T_G01_M05_C03_T028_F160_02_DY5EBB",
    "BIO_S1_SCS__1S_20260924T212535_20260924T212555_T_G01_M05_C04_T028_F160_02_DYAYBQ",
    "BIO_S1_SCS__1S_20260927T212537_20260927T212557_T_G01_M05_C05_T028_F160_02_DYGHV7",
    # Set 2, Region 2
    "BIO_S1_SCS__1S_20260915T090619_20260915T090640_T_G01_M05_C01_T020_F306_02_DXTDVD",
    "BIO_S1_SCS__1S_20260918T090622_20260918T090642_T_G01_M05_C02_T020_F306_02_DXZ169",
    "BIO_S1_SCS__1S_20260921T090624_20260921T090644_T_G01_M05_C03_T020_F306_02_DY4IY2",
    "BIO_S1_SCS__1S_20260924T090626_20260924T090646_T_G01_M05_C04_T020_F306_02_DYA1LO",
    "BIO_S1_SCS__1S_20260927T090628_20260927T090648_T_G01_M05_C05_T020_F306_02_DYFMCQ",
    "BIO_S1_SCS__1S_20260930T090630_20260930T090651_T_G01_M05_C06_T020_F306_02_DYL677",
]

bbox_south_america = [-81.5, -56.0, -34.0, 13.0]  # lists of floats, not strings
bbox_antarctica = [-180, -90, 180, -60]

bbox = None  # Set to valid bbox if filter with it
# Which products to load: "ids" (hard-coded list `selected`), "top_impact" (N largest iso_impact in products_rfi.json)
# or "threshold" (unmitigated products with iso_impact >= PRCT_IMPACT, from df_selected)
SELECT = "threshold"
N = 20
if SELECT == "top_impact":
    ids = df_rfi.nlargest(N, "iso_impact")["product_id"].to_list()
elif SELECT == "random":
    ids = df_rfi.sample(N)["product_id"].to_list()
elif SELECT == "threshold":
    ids = df_selected["product_id"].to_list()
else:
    ids = ecuador
print(len(ids))
if bbox is not None:
    search = catalog.search(
        method="GET",
        collections=["BiomassLevel1a"],
        datetime=["2026-01-01T00:00:00Z", "2026-07-29T23:33:00Z"],
        bbox=bbox,
        max_items=len(ids),
    )
    items = list(search.items())
    print(f"{len(items)} items accessed (max_items=10), {search.matched()} matched the query.")
    data = search.item_collection_as_dict()
else:
    B = 100
    searches = [
        catalog.search(method="POST", ids=ids[i : i + B], limit=B) for i in range(0, len(ids), B)
    ]
    items = [it for s in searches for it in s.items()]
    missing = set(ids) - {it.id for it in items}
    print(f"{len(items)} of {len(ids)} ids found, {len(missing)} missing.")
    # convert to df for easy visualisation
    data = {
        "type": "FeatureCollection",
        "features": [f for s in searches for f in s.items_as_dicts()],
    }

In [ ]:
df = pd.json_normalize(data, record_path=["features"])[
    [
        "id",  # unique id
        "properties.eopf:datatake_id",  # not unique id
        "properties.product:type",
        "properties.updated",
        "properties.sat:absolute_orbit",
        "properties.sat:orbit_state",
        # "assets.thumbnail.href",
        "assets.enclosure_kml.href",
        "assets.quicklook_jpg.href",
        "assets.enclosure_i_abs_tiff.href",
        "assets.enclosure_i_phase_tiff.href",
        "assets.product.href",
    ]
]

# Renaming the assets for
df.rename(
    columns={
        "properties.eopf:datatake_id": "dt_id",
        "properties.product:type": "product_type",
        "properties.updated": "last_modified",
        "properties.sat:absolute_orbit": "abs_orbit",
        "properties.sat:orbit_state": "orbit_state",
        # "assets.thumbnail.href": "thumbnail",
        "assets.enclosure_kml.href": "quicklook_kml",
        "assets.quicklook_jpg.href": "quicklook_jpg",
        "assets.enclosure_i_abs_tiff.href": "abs_product",
        "assets.enclosure_i_phase_tiff.href": "phase_product",
        "assets.product.href": "zipped_product",
    },
    inplace=True,
)

df.sort_values(by="id", ascending=True, ignore_index=True, inplace=True)
df

In [ ]:
by_id = {it.id: it for it in items}
items_selected = [by_id[i] for i in ids if i in by_id]
print(f"{len(items_selected)} products to process")

#### Run the next cell to select a smaller subset of items to process

In [ ]:
N = 10
items_selected = list(items)[:N]  # the first N
print(f"{len(items_selected)} products to process")

## Load the product

In [ ]:
from pathlib import Path

KEYS = ["enclosure_annot_xml", "enclosure_i_abs_tiff", "enclosure_i_phase_tiff", "quicklook_jpg"]

# Already downloaded: just resolve the local paths of each product
paths_by_id = {
    item.id: {k: data_path / item.id / Path(item.assets[k].href).name for k in KEYS}
    for item in items_selected
}

missing = [p for d in paths_by_id.values() for p in d.values() if not p.exists()]
print(len(paths_by_id), "products,", len(missing), "missing files")

In [ ]:
# 2) read one product on demand (not all at once, a full SCS is large)
POLS = ["HH", "HV", "VH", "VV"]  # assumed band order, check src.descriptions or the annotation XML

for item in items_selected:
    if item.id not in paths_by_id:
        continue  # failed download
    slc = read_item(item.id, paths_by_id)
    print(slc.shape)
    plot_product(
        slc,
        item.id,
        fetch_quicklook(item, paths_by_id[item.id]["quicklook_jpg"]),
        out_dir=f"{out_path}/{item.id}",
    )
    del slc

The product must have been downloaded in the plot_rfi_product notebook

Detect with Lambda-1 Detector and Supress the RFI with BSF
source: <https://github.com/huizhangyang/lambda1-bsf/tree/main>

### Single run with default hyperparameters

In [ ]:
from scipy.sparse.linalg import svds
from lambda1_bsf.yang_filter import get_eig_th
from rfi_mitigation import lambda1_bsf, block_matrix, freq_whiten


def ups_bdr(annot_xml):
    """ups/bdr for lambda1: axis 1 = azimuth, axis 2 = range."""
    root = ET.parse(annot_xml).getroot()

    def get(path):
        el = root.find(path)
        if el is None:
            raise KeyError(f"tag not found: {path}")
        return float(el.text)

    az, rg = "azimuthProcessingParameters", "rangeProcessingParameters"
    bdr = lambda blk: (lambda a: a**2 + (1 - a) ** 2 / 2)(
        get(f".//{blk}/windowCoefficient")
    )  # Hamming: B_eq/B
    return {
        # SLC azimuth sampling rate (= 1/azimuthTimeInterval) / B_az; not .//prf/value, the raw PRF
        "ups1": get(f".//{az}/totalBandwidth") / get(f".//{az}/processingBandwidth"),
        "ups2": get(f".//{rg}/totalBandwidth")
        / get(f".//{rg}/processingBandwidth"),  # fs_rg / B_rg
        "bdr1": bdr(az),
        "bdr2": bdr(rg),
    }

In [ ]:
from plotting import pauli_rgb

ids_pauli = ["BIO_S3_SCS__1S_20260829T233101_20260829T233122_T_G01_M04_C05_T044_F156_02_DWYO3C"]
MAX_PX = 4000  # decimation as in plot_result: box-average by n // MAX_PX per axis (None = full resolution)

for item in items_selected:
    if item.id not in ids_pauli or item.id not in paths_by_id:  # only the listed products that are downloaded
        continue
    slc = read_item(item.id, paths_by_id)  # (4, R, C), POLS order
    if slc is None:
        continue
    rgb = pauli_rgb(slc, max_px=MAX_PX)  # same Pauli image as the panels of plot_result
    del slc
    plt.imsave(f"{out_path}/pauli_{item.id}.png", rgb)  # one image pixel per SLC pixel: no axes, no margins
    plt.figure(figsize=(4, 14))
    plt.imshow(rgb, aspect="auto")
    plt.axis("off")
    plt.show()
    del rgb

In [ ]:
POLS = ["HH", "HV", "VH", "VV"]  # assumed band order, check src.descriptions or the annotation XML

narrow_search = True  # if True, only process a few selected products (see below), otherwise all products in items_selected
N_iterations = 1 # run the filter multiple times to see if it improves the result
full_pol = True  # if True, run the filter on all polarizations (plots pol-summed power + Pauli RGB), otherwise only on selected_pol
joint = False  # full pol only: True = the 4 pols spliced into one matrix per block, False = per-channel (same filter on each pol alone)
flatten = True # for freq only
if not full_pol:
    selected_pol = "HV"
    ch = POLS.index(selected_pol)

params = dict(lblk=(500, 500), lpd=(50, 50), gamma=1, ns_max=30, ns_mul=2, domain="image")  # tweak here

# params = dict(lblk=(500, 100), lpd=(50, 0), gamma=3, ns_max=30, ns_mul=2, domain="freq", flatten=True, bright=8)  # freq only: flatten = measured whitening, bright = bright-scatterer mask (sigma)

if narrow_search:
    # ids = ["BIO_S1_SCS__1S_20260915T212528_20260915T212549_T_G01_M05_C01_T028_F160_02_DXUDDE",
    #     "BIO_S1_SCS__1S_20260915T090619_20260915T090640_T_G01_M05_C01_T020_F306_02_DXTDVD"]
    # items_selected_aux = [by_id[i] for i in ids if i in by_id]
    # IDX = [0, 1, 4, 5, 6, 7]
    IDX = [6]
    items_selected_aux = [items_selected[i] for i in IDX]
else:
    items_selected_aux = items_selected

for item in tqdm(items_selected_aux):
    if item.id not in paths_by_id:
        continue
    p = ups_bdr(paths_by_id[item.id]["enclosure_annot_xml"])
    slc = read_item(item.id, paths_by_id)
    for idx in range(N_iterations):
        if full_pol:
            slc_clean, mask = lambda1_bsf(slc, **params, **p, norm=True, joint=joint)
        else:
            slc_clean, mask = lambda1_bsf(slc[ch], **params, **p)

        # rgb = pauli_rgb(slc_clean, max_px=4000)  # same Pauli image as the panels of plot_result
        # plt.imsave(f"{out_path}/pauli_mitigated_{item.id}.png", rgb)  # one image pixel per SLC pixel: no axes, no margins
        # plt.figure(figsize=(4, 14))
        # plt.imshow(rgb, aspect="auto")
        # plt.axis("off")
        # plt.show()
        # del rgb

        plot_result(
            slc if full_pol else slc[ch],  # full pol: (4, R, C) -> pol-summed power + Pauli RGB
            slc_clean,
            mask,
            max_px=2000,
            aspect="auto",
            title=f"{item.id} {('full pol ' + ('joint' if joint else 'per-channel')) if full_pol else selected_pol} iteration {idx + 1}",
            spectrum=True,
            out_dir=f"{out_path}/full_pol/{params['domain']}/{'joint' if joint else 'per-channel'}/{item.id}_lambda1_bsf_iteration{idx + 1}"
        )
        if N_iterations > 1:
            print(f"iteration {idx + 1} done")
            if full_pol:
                slc = slc_clean
            else:
                slc[ch] = slc_clean


    del slc, slc_clean, mask

### Sweep over hyperparameters

In [ ]:
import pandas as pd
from itertools import product
from scipy.ndimage import median_filter
from rasterio.windows import Window


def ie(x, edges):  # intensity entropy, lower = better
    h, _ = np.histogram(x, bins=edges)
    p = h[h > 0] / h.sum()
    return -(p * np.log2(p)).sum()


def ic(x):  # coefficient of variation of intensity
    return x.std() / x.mean()


def blind_metrics(slc, clean, mask):
    m = mask.astype(bool)
    if not m.any():
        return {}
    I0, I1 = np.abs(slc[..., m]) ** 2, np.abs(clean[..., m]) ** 2  # flagged pixels only
    edges = np.linspace(0, np.percentile(I0, 99.5), 129)  # same bins before/after
    p_rfi = (np.abs(slc[..., m] - clean[..., m]) ** 2).sum()
    return {
        "mnr_dB": 10 * np.log10(p_rfi / I1.sum()),  # P_removed / P_clean, lower = better
        "IE_gain": ie(I0, edges) - ie(I1, edges),  # >0 = entropy dropped
        "IC_ratio": ic(I1) / ic(I0),
    }


def row_rough(slc, k=101):
    """Streak metric: std of the row-mean dB profile minus its running median. Lower = fewer streaks."""
    p = (np.abs(slc) ** 2).mean(axis=-1).reshape(-1, slc.shape[-2]).mean(axis=0)  # one value per azimuth row (mean over pols)
    p = 10 * np.log10(p + 1e-12)
    return np.std(p - median_filter(p, k, mode="nearest"))


def metrics(slc, clean, mask, rough_src):
    p_src = (np.abs(slc) ** 2).sum()
    return {
        "flag_frac": mask.mean(),  # fraction of pixels in flagged blocks
        "removed_frac": (np.abs(slc - clean) ** 2).sum() / p_src,  # energy removed / total energy
        "rough_ratio": row_rough(clean) / rough_src,  # <1: streaks reduced
    }


grid = {
    "lblk": [(250, 250), (500, 500), (750, 750)],
    "lpd": [(50, 50)],
    "gamma": [1, 2.5, 5],
    "ns_mul": [1, 2],
    "ns_max": [15, 30],
    "domain": ["image"],
}
show = {(1, 250, 1, 15), (1, 250, 2, 30), (1, 500, 2, 30), (1, 750, 2, 30), (2.5, 500, 2, 30), (5, 500, 2, 30)}  # (gamma, lblk, ns_mul, ns_max) runs to inspect visually

rows = []

# N = 1
# items_selected_aux = list(items)[:N]  # the first N
# print(f"{len(items_selected_aux)} products to process")
# ids = ["BIO_S1_SCS__1S_20260915T212528_20260915T212549_T_G01_M05_C01_T028_F160_02_DXUDDE",
#        "BIO_S1_SCS__1S_20260915T090619_20260915T090640_T_G01_M05_C01_T020_F306_02_DXTDVD"]
# ids = ["BIO_S3_SCS__1S_20260829T233101_20260829T233122_T_G01_M04_C05_T044_F156_02_DWYO3C"]
items_selected_aux = [by_id[i] for i in ids if i in by_id]

POLS = ["HH", "HV", "VH", "VV"]  # assumed band order, check src.descriptions or the annotation XML

selected_pol = "HV"
ch = POLS.index(selected_pol)
full_pol = True  # if True, run the filter on all polarizations, otherwise only on selected_pol
joint = False  # full pol only: True = the 4 pols spliced into one matrix per block, False = per-channel (same filter on each pol alone)
window = Window(300, 2000, 400, 7500)  # None = whole product; a window also switches the plot to Pauli + removed RFI only
for item in tqdm(items_selected_aux):
    if item.id not in paths_by_id:
        continue
    slc = read_item(item.id, paths_by_id, window=window)  # (4, R, C)
    if slc is None:
        continue
    if not full_pol:
        slc = slc[ch][None]  # (1, R, C): same code path as full pol
    rough_src = row_rough(slc)
    for values in product(*grid.values()):
        params = dict(zip(grid, values))
        p = ups_bdr(paths_by_id[item.id]["enclosure_annot_xml"])
        qs = []
        clean, mask = lambda1_bsf(slc, **params, **p, q_out=qs, joint=joint)
        rows.append(
            {
                "id": item.id,
                **params,
                "mean_q": np.mean(qs) if qs else 0,
                **metrics(slc, clean, mask, rough_src),
                **blind_metrics(slc, clean, mask),
            }
        )
        if (params["gamma"], params["lblk"][0], params["ns_mul"], params["ns_max"]) in show:  # lblk[0]: show lists the block height as an int
            plot_result(
                slc if full_pol else slc[0],  # full pol: (4, R, C) -> pol-summed power + Pauli RGB
                clean if full_pol else clean[0],
                mask,
                max_px=2000,
                aspect="auto",
                title=f"gamma={params['gamma']} lblk={params['lblk']} ns_mul={params['ns_mul']} ns_max={params['ns_max']}",  # hyperparameters only; the product id is in the file name
                out_dir=f"{out_path}/full_pol/{params['domain']}/{'joint' if joint else 'per-channel'}/{item.id}_gamma{params['gamma']}_lblk{params['lblk']}_ns_mul{params['ns_mul']}_ns_max{params['ns_max']}.png",
                spectrum=True,
                only_pauli=full_pol and window is not None,
            )
        del clean, mask
    del slc

df = pd.DataFrame(rows)

In [ ]:
# Summary of the grid: one panel per metric vs gamma, one line per varying hyperparameter combination
LBLK_PLOT = 500  # block height to plot (None = all block sizes)
dfp = df if LBLK_PLOT is None else df[df.lblk.map(lambda b: b[0]) == LBLK_PLOT].copy()
assert len(dfp), f"no runs with lblk={LBLK_PLOT} in df"
vary = [k for k in ("lblk", "lpd", "ns_mul", "ns_max", "domain") if k in dfp and dfp[k].astype(str).nunique() > 1]
fmt = lambda k, v: f"{k}={v[0]}" if k in ("lblk", "lpd") and isinstance(v, tuple) else f"{k}={v}"
dfp["cfg"] = dfp.apply(lambda r: " · ".join(fmt(k, r[k]) for k in vary), axis=1) if vary else "default"

TITLES = {
    "flag_frac": "flagged fraction",
    "removed_frac": "removed energy fraction",
    "mnr_dB": "MNR [dB] (lower better)",
    "IE_gain": "entropy gain (> 0 better)",
    "IC_ratio": "IC ratio (< 1 better)",
    "mean_q": "mean q per flagged block",
}
REFS = {"IC_ratio": 1, "IE_gain": 0}  # no-change reference lines
cols = [c for c in TITLES if c in dfp]
n_pan, ncols = len(cols), 2
nrows = -(-n_pan // ncols)
cfgs = sorted(dfp.cfg.unique())
colors = {c: plt.get_cmap("tab20").colors[i % 20] for i, c in enumerate(cfgs)}
gammas = sorted(dfp.gamma.unique())

for pid, d in dfp.groupby("id"):
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.8 * ncols + 3.8, 2.6 * nrows), constrained_layout=True, squeeze=False)
    axes = axes.ravel()
    for a in axes[n_pan:]:
        a.set_visible(False)
    for cfg, g in d.groupby("cfg"):
        g = g.sort_values("gamma")
        for a, c in zip(axes, cols):
            a.plot(g.gamma, g[c], "o-", ms=4, lw=1.6, color=colors[cfg], label=cfg)
    for a, c in zip(axes, cols):
        a.set_title(TITLES[c], fontsize=14, loc="left")
        a.set_xlabel("γ", fontsize=13)
        a.set_xticks(gammas)
        if c in REFS:
            a.axhline(REFS[c], color="gray", lw=0.9, ls="--")
    for a in axes[:n_pan]:
        a.grid(alpha=0.3)
        a.tick_params(labelsize=12)
        a.spines[["top", "right"]].set_visible(False)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="outside right center", title=" · ".join(vary) or None, fontsize=12, title_fontsize=13, frameon=False)
    fig.suptitle(pid + ("" if LBLK_PLOT is None else f"   ·   lblk={LBLK_PLOT}"), fontsize=13, x=0.01, ha="left")
    plt.savefig(f"{out_path}/full_pol/{params['domain']}/{'joint' if joint else 'per-channel'}/{pid}_grid_summary.png", dpi=200, bbox_inches="tight")
    plt.show()


### Diagnostic

In [ ]:
def lam1(X):
    """Largest eigenvalue (s1^2) of the block, as in lamdba1_bsf."""
    return svds(X, k=1, maxiter=20)[1][0] ** 2


def cnoise(n, q, w, rng=np.random.default_rng(0)):
    """Circular complex Gaussian noise band-limited to the central fraction w=(w1, w2) of each axis."""
    X = np.fft.fft2(rng.standard_normal((n, q)) + 1j * rng.standard_normal((n, q)))
    f1, f2 = np.fft.fftfreq(n)[:, None], np.fft.fftfreq(q)[None, :]
    X *= (np.abs(f1) < w[0] / 2) & (np.abs(f2) < w[1] / 2)
    return np.fft.ifft2(X)


def prepare(slc, domain="image", norm=False, **kw):
    """Stack, threshold kwargs and outlier rule that the block loop of lambda1_bsf sees for this domain:
    image = the SLC itself, freq = the whitened in-band range spectrum (ups2 = bdr2 = 1, no outlier zeroing)."""
    slc = slc[None] if slc.ndim == 2 else slc
    if norm:
        slc = slc / np.sqrt((np.abs(slc) ** 2).mean(axis=(1, 2), keepdims=True))
    if domain == "freq":
        F, inb, w = freq_whiten(slc, kw["ups2"], kw["bdr2"])
        return F[..., inb] / w, {**kw, "ups2": 1, "bdr2": 1}, None
    return slc, kw, 8


def block_margins(stack, kw, outlier, lblk, lpd, min_size=100):
    """Per-block lambda1 / threshold (gamma=1) of a prepared stack. A block is flagged at `gamma` if margin > gamma."""
    _, R, C = stack.shape
    out = []
    for y in range(0, R, lblk[0]):
        for x in range(0, C, lblk[1]):
            X, _ = block_matrix(stack, y, x, lblk, lpd, outlier=outlier)
            if min(X.shape) < min_size:  # skip tiny edge slivers
                continue
            out.append((y, x, lam1(X) / get_eig_th(np.var(X), *X.shape, gamma=1, **kw)))
    return np.array(out)


def plot_spectrum(stack, kw, outlier, blocks, lblk, lpd, gammas=(1, 2.5, 10), top=100):
    """blocks: list of (y, x, label). Eigenvalues lambda = s^2 of the block matrix (pols spliced) vs the thresholds."""
    fig, ax = plt.subplots(
        1, len(blocks), figsize=(5 * len(blocks), 4), sharey=True, constrained_layout=True
    )
    for a, (y, x, label) in zip(np.atleast_1d(ax), blocks):
        X, _ = block_matrix(stack, y, x, lblk, lpd, outlier=outlier)
        lam = np.linalg.svd(X, compute_uv=False) ** 2
        n, q = X.shape
        var = np.var(X)
        k = min(top, len(lam))
        a.semilogy(np.arange(1, k + 1), lam[:k], "o", ms=3, color="C0")
        for g, c in zip(gammas, ("C1", "C2", "C3")):
            th = get_eig_th(var, n, q, gamma=g, **kw)
            a.axhline(th, ls="--", color=c, label=f"γ={g}: th, q={int((lam > th).sum())}")
        a.axhline(
            (np.sqrt(n) + np.sqrt(q)) ** 2 * var, ls=":", color="gray", label="white-noise MP edge"
        )
        a.set(title=f"{label}  (y={y}, x={x}, {n}x{q})", xlabel="eigenvalue index")
        a.legend(fontsize=7)
    np.atleast_1d(ax)[0].set_ylabel("λ = s²")
    plt.show()


from scipy.ndimage import gaussian_filter1d, uniform_filter1d


def streak_score(P, k_az=101, sigma=1.5):
    """Vertical-streak score of a power image P (azimuth line x range-frequency bin), freq domain: normalise each
    line by its median (scene brightness), smooth along azimuth (k_az lines, streaks persist, noise averages out),
    ridge filter along frequency only (difference of Gaussians, sigma = ridge half-width in bins), then the
    robust z-score (median / MAD) of the strongest response."""
    P = P / np.median(P, axis=1, keepdims=True)
    S = uniform_filter1d(P, k_az, axis=0)
    D = gaussian_filter1d(S, sigma, axis=1) - gaussian_filter1d(S, 4 * sigma, axis=1)
    d = D - np.median(D)
    return (d / (1.4826 * np.median(np.abs(d)))).max()


def block_streaks(stack, lblk, lpd, min_size=50, **kw):
    """(y, x, streak_score) per block of a prepared freq-domain stack, same blocks as block_margins."""
    n, R, C = stack.shape
    out = []
    for y in range(0, R, lblk[0]):
        for x in range(0, C, lblk[1]):
            X, _ = block_matrix(stack, y, x, lblk, lpd, outlier=None)
            if min(X.shape) < min_size:
                continue
            P = (np.abs(np.stack(np.split(X, n, axis=1))) ** 2).sum(axis=0)   # power summed over pols
            out.append((y, x, streak_score(P, **kw)))
    return np.array(out)


DOMAIN = "image"  # "image" or "freq": same blocks as lambda1_bsf(domain=...)
FULL_POL = True  # True: the 4 pol patches are spliced into one matrix per block, False: only selected_pol
NORM = True  # as lambda1_bsf(norm=True)
LBLK, LPD = ((500, 500), (50, 50)) if DOMAIN == "image" else ((500, 100), (50, 0))

for item in tqdm(items_selected):
    if item.id not in paths_by_id:
        continue

    print(f"Processing product {item.id}")
    p = ups_bdr(paths_by_id[item.id]["enclosure_annot_xml"])
    print(p)
    slc = read_item(item.id, paths_by_id)
    if slc is None:
        continue
    if not FULL_POL:
        slc = slc[POLS.index(selected_pol)][None]
    stack, kw, outlier = prepare(slc, DOMAIN, NORM, **p)
    del slc
    n_pol = stack.shape[0]
    mm = block_margins(stack, kw, outlier, LBLK, LPD, min_size=100 if DOMAIN == "image" else 50)
    m_real = mm[:, 2]

    w = (kw["bdr1"] / kw["ups1"], kw["bdr2"] / kw["ups2"])  # occupied band fraction per axis (1 = white, freq domain)
    shape = (LBLK[0] + 2 * LPD[0], LBLK[1] + 2 * LPD[1])  # an interior block, per pol
    m_sim = np.array(
        [
            lam1(X) / get_eig_th(np.var(X), *X.shape, gamma=1, **kw)
            for X in (
                np.concatenate([cnoise(*shape, w) for _ in range(n_pol)], axis=1) for _ in range(100)
            )
        ]
    )

    bins = np.linspace(np.log10(min(m_real.min(), m_sim.min())), np.log10(m_real.max()), 40)
    plt.hist(np.log10(m_sim), bins, alpha=0.6, label="simulated H0 noise", density=True)
    plt.hist(
        np.log10(m_real), bins, alpha=0.6, label=f"real blocks (n={len(m_real)})", density=True
    )
    for g in (1, 2.5, 10):
        plt.axvline(np.log10(g), color="k", ls="--", lw=0.8)
        plt.text(np.log10(g), plt.ylim()[1] * 0.9, f"γ={g}", fontsize=8)
    plt.xlabel("log10(λ1 / threshold at γ=1)")
    plt.legend()
    plt.show()
    print(
        {g: round((m_real > g).mean(), 3) for g in (1, 2.5, 5, 10, 20)}
    )  # blocks flagged per gamma
    print("sim max margin:", m_sim.max())

    if DOMAIN == "freq":  # vertical-streak statistic next to lambda1: does it separate RFI from scene structure?
        ss = block_streaks(stack, LBLK, LPD)
        s_sim = np.array(
            [
                streak_score(sum(np.abs(cnoise(*shape, w)) ** 2 for _ in range(n_pol)))
                for _ in range(100)
            ]
        )
        fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
        bins = np.linspace(min(ss[:, 2].min(), s_sim.min()), max(ss[:, 2].max(), s_sim.max()), 40)
        ax[0].hist(s_sim, bins, alpha=0.6, label="simulated H0 noise", density=True)
        ax[0].hist(ss[:, 2], bins, alpha=0.6, label=f"real blocks (n={len(ss)})", density=True)
        ax[0].set(xlabel="vertical-streak score (robust z of strongest ridge)")
        ax[0].legend()
        margin = {(int(y), int(x)): m for y, x, m in mm}
        ax[1].scatter(ss[:, 2], [np.log10(margin[(int(y), int(x))]) for y, x, _ in ss], s=6)
        ax[1].set(xlabel="vertical-streak score", ylabel="log10(λ1 / threshold at γ=1)")
        plt.show()

    o = np.argsort(mm[:, 2])
    pick = [
        (int(mm[i, 0]), int(mm[i, 1]), lab)
        for i, lab in [
            (o[0], "lowest margin"),
            (o[len(o) // 2], "median"),
            (o[-1], "highest margin"),
        ]
    ]
    plot_spectrum(stack, kw, outlier, pick, LBLK, LPD)

In [ ]:
# Per-pol comparison on one product: per-channel vs joint (4 pols spliced) vs joint + norm, same parameters
from scipy.ndimage import median_filter as _mf
from scipy.signal import find_peaks


def mean_spec_db(S, chunk=2000):
    """(n, R, C) -> (n, C): mean range power spectrum per pol in dB, fft-shifted (chunked, S can be large)."""
    P = 0
    for r in range(0, S.shape[1], chunk):
        P = P + (np.abs(np.fft.fft(S[:, r : r + chunk], axis=-1)) ** 2).sum(axis=1)
    return 10 * np.log10(np.fft.fftshift(P / S.shape[1], axes=-1) + 1e-20)


def find_lines(spec_db, ups2, k=3, win=101):
    """Bins of the k strongest narrowband lines in the pol-pooled spectrum (excess over a running median)."""
    pooled = 10 * np.log10((10 ** (spec_db / 10)).mean(axis=0))
    exc = pooled - _mf(pooled, win, mode="nearest")
    exc[np.abs(np.fft.fftshift(np.fft.fftfreq(len(pooled)))) >= 0.5 / ups2] = -np.inf  # in-band only
    pk, _ = find_peaks(exc, distance=10)
    return pk[np.argsort(exc[pk])[::-1][:k]]


def line_excess_db(spec_db, bins, win=101):
    """Mean height [dB] of the tracked lines above the running-median baseline of the spectrum, per pol."""
    return (spec_db - _mf(spec_db, size=(1, win), mode="nearest"))[:, bins].mean(axis=1)


def coh(a, b):
    """Complex coherence magnitude of two channels."""
    return np.abs((a * b.conj()).sum()) / np.sqrt((np.abs(a) ** 2).sum() * (np.abs(b) ** 2).sum())


def compare_variants(slc, p, lblk, lpd, domain, n_lines=3, **params):
    """Per-pol metrics of per-channel / joint / joint+norm filtering of slc (4, R, C), same params for all.
    Returns (per_pol table, per_variant table). Each variant is dropped after its metrics (a full SCS is large)."""
    spec0 = mean_spec_db(slc)
    bins = find_lines(spec0, p["ups2"], n_lines)
    print("tracked lines, normalized range frequency:", np.round(np.fft.fftshift(np.fft.fftfreq(slc.shape[-1]))[bins], 4))
    e0 = line_excess_db(spec0, bins)
    p_src = (np.abs(slc) ** 2).sum(axis=(1, 2))
    per_pol = [dict(variant="original", pol=k, removed_frac=0.0, line_dB_before=e0[i], line_dB_after=e0[i], suppression_dB=0.0)
               for i, k in enumerate(POLS)]
    per_var = [dict(variant="original", HH_VV_coh=coh(slc[0], slc[3]), HV_VH_corr=coh(slc[1], slc[2]), mean_q=np.nan, at_cap=np.nan)]
    for name in ("per-channel", "joint", "joint+norm"):
        qs = []
        kw = dict(lblk=lblk, lpd=lpd, domain=domain, q_out=qs, **params, **p)
        if name == "per-channel":
            out = lambda1_bsf(slc, joint=False, **kw)[0]
        else:
            out = lambda1_bsf(slc, norm=(name == "joint+norm"), **kw)[0]
        e1 = line_excess_db(mean_spec_db(out), bins)
        rem = (np.abs(slc - out) ** 2).sum(axis=(1, 2)) / p_src
        per_pol += [dict(variant=name, pol=k, removed_frac=rem[i], line_dB_before=e0[i], line_dB_after=e1[i], suppression_dB=e0[i] - e1[i])
                    for i, k in enumerate(POLS)]
        q = np.array(qs)
        per_var.append(dict(variant=name, HH_VV_coh=coh(out[0], out[3]), HV_VH_corr=coh(out[1], out[2]),
                            mean_q=q.mean() if q.size else 0, at_cap=(q == params["ns_max"]).mean() if q.size else 0))
        del out
    return pd.DataFrame(per_pol), pd.DataFrame(per_var)


ITEM_IDX = [0, 1, 4, 5, 7]
CMP_DOMAIN = "freq"  # "image" or "freq"
CMP_PARAMS = dict(gamma=1.5, ns_max=30, ns_mul=2, flatten=True, bright=8)  # flatten/bright only act in the freq domain

for idx in tqdm(ITEM_IDX):
    item = items_selected[idx]
    if item.id not in paths_by_id:
        continue
    print(f"Processing product {item.id}")
    slc = read_item(item.id, paths_by_id)  # (4, R, C), POLS order
    p = ups_bdr(paths_by_id[item.id]["enclosure_annot_xml"])
    lblk, lpd = ((500, 500), (50, 50)) if CMP_DOMAIN == "image" else ((500, 100), (50, 0))
    per_pol, per_var = compare_variants(slc, p, lblk, lpd, CMP_DOMAIN, **CMP_PARAMS)
    del slc
    print(item.id)
    order = ["original", "per-channel", "joint", "joint+norm"]
    display(per_pol.pivot(index="variant", columns="pol", values=["removed_frac", "suppression_dB"]).loc[order, :].round(3))
    display(per_var.round(3))


In [ ]:
import re

# False-alarm rate on RFI-free products: sweep gamma (only the detector runs, nothing is removed)
CLEAN_IDS = [  # edit: products that you checked by hand to be RFI-free
    "BIO_S3_SCS__1S_20260829T233255_20260829T233313_T_G01_M04_C05_T044_F162_02_DWYO99",
    "BIO_S3_SCS__1S_20260829T233236_20260829T233257_T_G01_M04_C05_T044_F161_02_DWYO8H",
]
# same options as the filter run that is being validated
FA_PARAMS = dict(lblk=(500, 500), lpd=(50, 50), domain="image", joint=False, norm=False, flatten=False, bright=None)
GAMMAS = np.round(np.arange(1.0, 3.01, 0.1), 2)
TARGETS = (0.01, 0.001)  # false-alarm rates to reach
GAMMA_REF = 2.0  # gamma at which each product is listed in the right panel
FA_WINDOW = None  # rasterio Window to read only a part of each product, None = whole product

# lambda1 / threshold(gamma = 1) of every block: a block is flagged at gamma if its margin > gamma
margins = {}
for item in tqdm(items_selected):
    slc = read_item(item.id, paths_by_id, window=FA_WINDOW)  # (4, R, C)
    if slc is None:
        continue
    m = []
    lambda1_bsf(slc, gamma=1.0, margins=m, detect_only=True, **FA_PARAMS, **ups_bdr(paths_by_id[item.id]["enclosure_annot_xml"]))
    margins[item.id] = np.asarray(m)
    del slc
assert margins, "none of CLEAN_IDS is available (check by_id / paths_by_id)"

short = lambda pid: re.search(r"\d{8}T\d{6}", pid).group() + "_" + re.search(r"_(F\d+)_", pid).group(1)  # start time + frame
far = pd.DataFrame({short(k): [(v > g).mean() for g in GAMMAS] for k, v in margins.items()}, index=GAMMAS)
allm = np.concatenate(list(margins.values()))
far["pooled"] = [(allm > g).mean() for g in GAMMAS]
far.index.name = "gamma"

print("blocks per product:", {short(k): len(v) for k, v in margins.items()}, "| pooled:", len(allm))
for t in TARGETS:  # smallest gamma that reaches the target, pooled and for the worst product
    ok_pooled = far.index[far["pooled"] <= t]
    ok_each = far.index[(far.drop(columns="pooled") <= t).all(axis=1)]
    print(f"false-alarm rate <= {t:.1%}:  pooled gamma >= {ok_pooled[0] if len(ok_pooled) else 'not reached'}"
          f"  |  every product gamma >= {ok_each[0] if len(ok_each) else 'not reached'}")

prod = far.drop(columns="pooled")
g_ref = far.index[np.abs(far.index.values - GAMMA_REF).argmin()]  # gamma at which every product is listed on the right
opts = f"{FA_PARAMS['domain']} domain, {'joint' if FA_PARAMS['joint'] else 'per-channel'}, lblk={FA_PARAMS['lblk'][0]}" + (
    ", flatten" if FA_PARAMS.get("flatten") else "") + (", bright mask" if FA_PARAMS.get("bright") else "")

fig, (ax, bx) = plt.subplots(1, 2, figsize=(14, 6.2), gridspec_kw=dict(width_ratios=[1.3, 1]), constrained_layout=True)

# left: false-alarm rate vs gamma, pooled on top of the spread of the individual products
for i, col in enumerate(prod):
    ax.plot(far.index, 100 * prod[col], color="tab:blue", lw=0.7, alpha=0.3, label="each product" if i == 0 else None)
ax.fill_between(far.index, 100 * prod.quantile(0.1, axis=1), 100 * prod.quantile(0.9, axis=1), color="tab:blue", alpha=0.18,
                lw=0, label="10–90 % of products")
ax.plot(far.index, 100 * prod.median(axis=1), color="tab:blue", lw=2, label="median product")
ax.plot(far.index, 100 * far["pooled"], color="k", lw=2.8, label="pooled")
for t in TARGETS:
    ax.axhline(100 * t, color="gray", lw=0.9, ls="--")
    ax.text(1.01, 100 * t, f"{t:.1%}", transform=ax.get_yaxis_transform(), va="center", fontsize=11, color="gray")
    ok = far.index[far["pooled"] <= t]
    if len(ok):  # first gamma where the pooled rate reaches the target
        ax.plot(ok[0], 100 * far.loc[ok[0], "pooled"], "o", ms=9, mfc="white", mec="k", mew=2, zorder=5)
        ax.annotate(f"γ = {ok[0]:g}", (ok[0], 100 * far.loc[ok[0], "pooled"]), xytext=(8, 10), textcoords="offset points",
                    fontsize=11, fontweight="bold")
ax.set_yscale("symlog", linthresh=0.1)  # 0 stays visible
ax.set_ylim(0, max(100, 100 * far.values.max() * 1.2))
ax.set_yticks([0, 0.1, 1, 10, 100])
ax.set_yticklabels(["0", "0.1", "1", "10", "100"])
ax.minorticks_off()
ax.set_xlabel("γ", fontsize=13)
ax.set_ylabel("blocks flagged [%]", fontsize=13)
ax.set_title("false-alarm rate vs γ", fontsize=13, loc="left")
ax.legend(fontsize=11, frameon=False, loc="upper right")

# right: which products drive it, at gamma = g_ref
s = (100 * prod.loc[g_ref]).sort_values()
bx.barh(range(len(s)), s.values, color=np.where(s.values > 100 * TARGETS[0], "tab:red", "tab:blue"), height=0.7)
bx.axvline(100 * TARGETS[0], color="gray", lw=0.9, ls="--")
bx.set_yticks(range(len(s)))
bx.set_yticklabels(s.index, fontsize=8)
bx.set_xlabel("blocks flagged [%]", fontsize=13)
bx.set_title(f"per product at γ = {g_ref:g}", fontsize=13, loc="left")

for a in (ax, bx):
    a.tick_params(axis="x", labelsize=12)
    a.grid(alpha=0.3, axis="x" if a is bx else "both")
    a.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", labelsize=12)
fig.suptitle(f"False-alarm rate on RFI-free products   ·   {len(prod.columns)} products, {len(allm)} blocks   ·   {opts}",
             fontsize=13, x=0.01, ha="left")
fig.savefig(f"{out_path}/false_alarm_rate.png", dpi=200, bbox_inches="tight")
plt.show()
far.round(4)
